# Liputan6 -> AMR (Indonesian -> Indonesian AMR)

Parses Liputan6 sentences into AMR graphs using **Abdi's ID->ID parser**:
`abdiharyadi/mbart-en-id-smaller-indo-amr-parsing-translated-nafkhan`
(Smatch 0.8299).

**No translation step.** The model takes plain Indonesian. Verified from the
model repo's own `dummy_input.json`, whose reference input is:

```
Kami...</s> <AMR> <mask> </AMR> <pad><pad><pad>
```

i.e. `<indonesian text> </s> <AMR> <mask> </AMR>` - no `id_ID`, no `en_XX`,
no English. Section 4 below *verifies* this by reproducing those exact token ids.

## Setup
1. **Add Input:** `amr-code-modules` (the `common/` + `model_interface/` folders)
2. **Add Input:** `liputan6-data` (`analysis_data.csv`)
3. **Accelerator:** GPU **T4**
4. **Internet: ON** - the model is downloaded from HuggingFace
5. **Persistence: Files only**

## How to run
This notebook builds a Python 3.10 conda env in Cell 1 and you must **switch the
kernel by hand** afterwards, so a headless *Save & Run All* will not work.
Run interactively, then click **Save Version** yourself when it finishes.

# 1. Python 3.10 environment

In [ ]:
%%bash
PYTHON_VERSION="3.10"
ENV_NAME="amr_env"
ENV_PATH="/kaggle/working/$ENV_NAME"
echo "=== conda env with Python $PYTHON_VERSION ==="
eval "$(conda shell.bash hook)"
if [ ! -d "$ENV_PATH" ]; then
    conda create -y -p "$ENV_PATH" python=$PYTHON_VERSION
else
    echo "Environment already exists at $ENV_PATH"
fi
conda activate "$ENV_PATH"
pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# transformers pinned to the version the model was trained with (see its README)
pip install --quiet "transformers==4.44.0" "penman>=1.1.0" sentencepiece sacremoses \
    regex networkx pandas tqdm ipykernel huggingface_hub
python -m ipykernel install --user --name=$ENV_NAME --display-name "Python ($PYTHON_VERSION) AMR"
echo "=== done ==="
echo ">>> Kernel > Change Kernel > 'Python (3.10) AMR', then continue at Cell 2."

## After Cell 1: **Kernel -> Change Kernel -> 'Python (3.10) AMR'**, then continue below.

# 2. Imports & paths

In [ ]:
import os, sys, json, shutil, time, torch, penman, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================================
# PATHS - Kaggle mounts datasets as
#   /kaggle/input/datasets/<username>/<dataset-slug>/
# ============================================================
BASE              = "/kaggle/input/datasets/fedrianzdharma"
CODE_MODULES_PATH = f"{BASE}/amr-code-modules"
DATA_PATH         = f"{BASE}/liputan6-data"

# Model is pulled straight from HuggingFace (needs Internet ON) - no dataset upload.
MODEL_ID          = "abdiharyadi/mbart-en-id-smaller-indo-amr-parsing-translated-nafkhan"

OUTPUT_DIR        = "/kaggle/working/amr_graphs"

# CROSS-SESSION RESUME: save each run's output as a dataset named
# "liputan6-amr-graphs", then it mounts here. Use "" on the FIRST run.
PREV_DIR          = f"{BASE}/liputan6-amr-graphs/amr_graphs"
# PREV_DIR = ""   # <- first run

# ============================================================
# SCOPE - assignment target is ~10rb (10,000) TRAINING documents.
#   SPLIT     : which Liputan6 split to parse ("train"/"test"/None=all).
#               MUST be set - the csv is sorted with test BEFORE train, so
#               taking "the first N rows" silently gives you the test set.
#   DOC_LIMIT : cap on DOCUMENTS parsed (all sentences of a doc kept
#               together). None = no cap.
# ============================================================
SPLIT         = "train"
DOC_LIMIT     = 10000

# Safety rails so an interactive session ends cleanly and you can Save Version.
MAX_NEW       = 40000     # stop after this many NEW graphs this run (None = no cap)
TIME_BUDGET_H = 8.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
if CODE_MODULES_PATH not in sys.path:
    sys.path.insert(0, CODE_MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus

print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
for name, p in [("code modules", CODE_MODULES_PATH), ("data", DATA_PATH),
                ("prev output", PREV_DIR)]:
    print(f"{name:13s} exists: {os.path.exists(p) if p else False}  ({p})")

# 3. Load model & tokenizer (from HuggingFace)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

amr_config    = AutoConfig.from_pretrained(MODEL_ID)
amr_tokenizer = AMRBartTokenizer.from_pretrained(MODEL_ID, use_fast=False)
amr_model     = MBartForConditionalGeneration.from_pretrained(MODEL_ID, config=amr_config)
amr_model.resize_token_embeddings(len(amr_tokenizer))
amr_model     = amr_model.to(device)
amr_model.eval()

print("Device     :", amr_model.device)
print("vocab_size :", amr_config.vocab_size, "(expect 38025)")
print("<AMR> id   :", amr_tokenizer.amr_bos_token_id,
      "| </AMR> id:", amr_tokenizer.amr_eos_token_id,
      "| <mask> id:", amr_tokenizer.mask_token_id)

# 4. VERIFY the input format

The model repo ships `dummy_input.json` containing reference `input_ids` for five
Indonesian sentences. We rebuild those inputs with our own tokenization and assert
the ids match exactly.

**If this passes, the input format is provably correct** - plain Indonesian, no
language prefix, no translation. If it fails, do not run the full job: the printed
diff shows what our pipeline produced versus what the model expects.

In [ ]:
from huggingface_hub import hf_hub_download

dummy_path = hf_hub_download(repo_id=MODEL_ID, filename="dummy_input.json")
dummy = json.load(open(dummy_path, encoding="utf-8"))
pad_id = amr_tokenizer.pad_token_id

def build_input_ids(text):
    """Our pipeline: plain Indonesian text -> ids + <AMR> <mask> </AMR>."""
    ids = amr_tokenizer(text, max_length=None, truncation=True)["input_ids"]
    return ids + [amr_tokenizer.amr_bos_token_id,
                  amr_tokenizer.mask_token_id,
                  amr_tokenizer.amr_eos_token_id]

# Recover the raw sentence from the reference token string
SUFFIX = "</s> <AMR> <mask> </AMR>"
n_pass = 0
for tok_str, ref_ids in zip(dummy["input_tokens"], dummy["input_ids"]):
    text = tok_str.replace("<pad>", "").strip()
    text = text[:text.index(SUFFIX)].replace("</s>", "").strip() if SUFFIX in text else text
    expected = [t for t in ref_ids if t != pad_id]
    ours     = build_input_ids(text)
    ok = ours == expected
    n_pass += ok
    print(("PASS  " if ok else "FAIL  ") + repr(text))
    if not ok:
        print("   expected:", expected)
        print("   ours    :", ours)
        print("   exp toks:", amr_tokenizer.convert_ids_to_tokens(expected))
        print("   our toks:", amr_tokenizer.convert_ids_to_tokens(ours))

print(f"\n{n_pass}/{len(dummy['input_ids'])} matched the model's reference input.")
assert n_pass == len(dummy["input_ids"]), \
    "Input format mismatch - fix build_input_ids before running the full job."

# 5. Dataset & DataLoader

In [ ]:
class Liputan6Dataset(Dataset):
    """One row per SENTENCE (id = '{doc_id}_{sent_idx}').

    Keeps only sentences not yet parsed, looking at both this run's output and
    the previous run's mounted output so the job resumes across sessions.
    """
    def __init__(self, data_path=DATA_PATH, output_dir=OUTPUT_DIR,
                 doc_limit=DOC_LIMIT, split=SPLIT):
        df = pd.read_csv(os.path.join(data_path, "analysis_data.csv"), dtype={"id": str})
        total_rows, total_docs = len(df), df["doc_id"].nunique()

        if split is not None:
            if "split" not in df.columns:
                raise KeyError("csv has no 'split' column - regenerate it with "
                               "liputan6_to_csv.py so train/test are not mixed")
            print("Available splits    :", df["split"].value_counts().to_dict())
            df = df[df["split"] == split]
            if df.empty:
                raise ValueError(f"no rows for split={split!r}")

        if doc_limit is not None:
            keep = df["doc_id"].drop_duplicates().head(doc_limit)
            df = df[df["doc_id"].isin(set(keep))]

        parsed = set()
        for d in [PREV_DIR, output_dir]:
            if d and os.path.isdir(d):
                for fn in os.listdir(d):
                    if fn.endswith(".txt"):
                        parsed.add(fn[:-4])

        in_scope = len(df)
        self.df = df[~df["id"].isin(parsed)].reset_index(drop=True)

        print(f"Corpus              : {total_rows} sentences / {total_docs} docs")
        print(f"In scope ({split}, limit {doc_limit}): {in_scope} sentences / "
              f"{df['doc_id'].nunique()} docs")
        print(f"Already parsed      : {in_scope - len(self.df)}")
        print(f"Remaining to parse  : {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"id": row["id"], "text": str(row["text"])}


def collate_fn(batch):
    """Plain Indonesian text -> ids + <AMR> <mask> </AMR>, then left-aligned pad.

    Format verified against the model's dummy_input.json in section 4.
    """
    ids   = [b["id"] for b in batch]
    built = [build_input_ids(b["text"]) for b in batch]
    width = max(len(x) for x in built)
    pad   = amr_tokenizer.pad_token_id
    input_ids = [x + [pad] * (width - len(x)) for x in built]
    masks     = [[1] * len(x) + [0] * (width - len(x)) for x in built]
    return ids, input_ids, masks


ds     = Liputan6Dataset()
loader = DataLoader(ds, batch_size=1, collate_fn=collate_fn)

# 6. Decoding & storage

In [ ]:
def decode_amr_output(pred_token_ids, tokenizer):
    pred_ids = list(pred_token_ids)
    pred_ids[0] = tokenizer.bos_token_id
    pred_ids = [tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
                for tok in pred_ids if tok != tokenizer.pad_token_id]
    graph, status, _ = tokenizer.decode_amr(pred_ids, restore_name_ops=False)
    return penman.encode(graph), status


def store_graph(ids, amr_strings, output_dir=OUTPUT_DIR):
    for data_id, amr_str in zip(ids, amr_strings):
        with open(os.path.join(output_dir, f"{data_id}.txt"), "w", encoding="utf-8") as f:
            f.write(amr_str)

# 7. Smoke test - parse 3 sentences and look at them

Read these before launching the full run. The concepts should reflect the
sentence's meaning and the status should be `OK`; a `BACKOFF` means the decoder
produced something unparseable.

In [ ]:
for i in range(min(3, len(ds))):
    sample = ds[i]
    _, inp, msk = collate_fn([sample])
    with torch.no_grad():
        out = amr_model.generate(
            input_ids=torch.tensor(inp, dtype=torch.long).to(device),
            attention_mask=torch.tensor(msk, dtype=torch.long).to(device),
            num_beams=5, max_length=1024,
            decoder_start_token_id=amr_tokenizer.amr_bos_token_id)
    amr_str, status = decode_amr_output(out[0].cpu().tolist(), amr_tokenizer)
    print(f"--- {sample['id']} | status={getattr(status, 'name', status)}")
    print(sample["text"])
    print(amr_str)
    print()

# 8. Parse everything in scope

In [ ]:
start_t  = time.time()
budget_s = TIME_BUDGET_H * 3600
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed, stop_reason = 0, "completed all remaining"
print(f"Parsing {len(ds)} sentences...")

pbar = tqdm(loader, desc="Parsing AMR")
for batch_ids, inputs, masks in pbar:
    if MAX_NEW is not None and total_parsed >= MAX_NEW:
        stop_reason = f"hit MAX_NEW={MAX_NEW}"
        break
    if time.time() - start_t > budget_s:
        stop_reason = f"hit TIME_BUDGET_H={TIME_BUDGET_H}"
        break
    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=5, max_length=1024,
                decoder_start_token_id=amr_tokenizer.amr_bos_token_id)
        amr_strings = []
        for i in range(outputs.shape[0]):
            amr_string, status = decode_amr_output(outputs[i].cpu().tolist(), amr_tokenizer)
            amr_strings.append(amr_string)
            name = getattr(status, "name", str(status))
            status_counts[name if name in status_counts else "ERROR"] += 1
        store_graph(batch_ids, amr_strings)
        total_parsed += len(batch_ids)
    except Exception:
        status_counts["ERROR"] += len(batch_ids)
        continue
    if total_parsed % 500 == 0:
        torch.cuda.empty_cache()
    pbar.set_postfix(new=total_parsed, h=f"{(time.time()-start_t)/3600:.2f}")

pbar.close()
print("")
print(f"Stopped because : {stop_reason}")
print(f"Parsed this run : {total_parsed}")
print(f"Elapsed         : {(time.time()-start_t)/3600:.2f} h")
for k, v in status_counts.items():
    if v:
        print(f"  {k:8s}: {v} ({v/max(total_parsed,1)*100:.1f}%)")

# 9. Carry forward previous output, then zip

In [ ]:
copied = 0
if PREV_DIR and os.path.isdir(PREV_DIR):
    have = set(os.listdir(OUTPUT_DIR))
    for fn in tqdm(os.listdir(PREV_DIR), desc="Carrying forward"):
        if fn.endswith(".txt") and fn not in have:
            shutil.copyfile(os.path.join(PREV_DIR, fn), os.path.join(OUTPUT_DIR, fn))
            copied += 1

total = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")])
print(f"Carried forward : {copied}")
print(f"TOTAL graphs    : {total}")
print("\n>>> Click 'Save Version' - /kaggle/working is wiped when the session ends.")
print(">>> Then save the output as a dataset named 'liputan6-amr-graphs'.")

In [ ]:
import zipfile
zip_path = "/kaggle/working/liputan6_amr_graphs.zip"
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in tqdm(files, desc="Zipping"):
        zf.write(os.path.join(OUTPUT_DIR, fn), fn)
print(f"{zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB), {len(files)} graphs")